# Advanced Python Descriptors — Problems with Complete Solutions

This notebook develops advanced practice around **descriptors used as instance properties**.

Core ideas covered:

- `__get__`, `__set__`, and `__delete__`
- data vs non-data descriptors
- storing values on the instance
- avoiding storage-name collisions
- `__set_name__`
- interaction with `__dict__`
- interaction with `__slots__`
- descriptor reuse across classes
- per-instance state
- memory leaks caused by strong references
- `weakref.WeakKeyDictionary`
- hashability and weak-reference limitations
- validation descriptors
- coercion vs validation
- inheritance and descriptor lookup
- descriptor introspection
- computed/cached descriptors
- thread-safety considerations
- reusable descriptor frameworks

Each problem is followed by a full solution and executable checks.

## 0. Warm-up: Why the naïve storage design fails

A descriptor object usually lives on the **class**, so the same descriptor instance is shared by every instance of that class.

If the descriptor stores a value directly on itself, all instances may accidentally share the same value.

In [1]:
class BadInteger:
    def __init__(self):
        self.value = None

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return self.value

    def __set__(self, instance, value):
        self.value = int(value)


class Point:
    x = BadInteger()


p1 = Point()
p2 = Point()

p1.x = 10.9
print("p1.x:", p1.x)
print("p2.x:", p2.x)  # shared state

p1.x: 10
p2.x: 10


### Observation

The descriptor instance `Point.__dict__["x"]` is shared. Therefore, descriptor-local mutable state is class-level state unless that state is keyed by each managed instance.

# Problem 1 — Repair a storage collision between two descriptors

Consider a descriptor that always stores its value under the same instance attribute:

```python
class IntegerValue:
    def __set__(self, instance, value):
        instance.stored_value = int(value)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, "stored_value", None)
```

A class uses it twice:

```python
class Point2D:
    x = IntegerValue()
    y = IntegerValue()
```

## Tasks

1. Explain exactly why `x` and `y` overwrite each other.
2. Redesign the descriptor so that each managed attribute receives a distinct storage name automatically.
3. Do **not** require `IntegerValue("x")` and `IntegerValue("y")`; the descriptor should discover its assigned attribute name.
4. Verify the instance dictionary.

## Solution 1

`__set_name__(owner, name)` is invoked when the owner class is created. It lets the descriptor learn the attribute name under which it was assigned.

In [2]:
class IntegerValue:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name, None)

    def __set__(self, instance, value):
        setattr(instance, self.storage_name, int(value))


class Point2D:
    x = IntegerValue()
    y = IntegerValue()


p = Point2D()
p.x = 10.8
p.y = 20.9

assert p.x == 10
assert p.y == 20
assert p.__dict__ == {"_x": 10, "_y": 20}
print(p.x, p.y)
print(p.__dict__)
print(Point2D.x.public_name, Point2D.x.storage_name)
print(Point2D.y.public_name, Point2D.y.storage_name)

10 20
{'_x': 10, '_y': 20}
x _x
y _y


# Problem 2 — Build a strict typed descriptor

Create a reusable descriptor named `Typed` with these requirements:

- It must use `__set_name__`.
- It must store each value under a private instance name.
- It must reject values of the wrong type.
- It must **not coerce** values.
- Access through the class must return the descriptor object.
- Subclasses should define `expected_type`.

Then implement `Integer`, `Float`, and `String`, and use them in a `Product` class.

## Solution 2

In [3]:
class Typed:
    expected_type = object

    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return getattr(instance, self.storage_name)
        except AttributeError:
            raise AttributeError(
                f"{type(instance).__name__}.{self.public_name} has not been initialized"
            ) from None

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.public_name} must be {self.expected_type.__name__}; "
                f"got {type(value).__name__}"
            )
        setattr(instance, self.storage_name, value)


class Integer(Typed):
    expected_type = int

class Float(Typed):
    expected_type = float

class String(Typed):
    expected_type = str


class Product:
    sku = String()
    quantity = Integer()
    price = Float()

    def __init__(self, sku, quantity, price):
        self.sku = sku
        self.quantity = quantity
        self.price = price


item = Product("ABC-001", 8, 19.95)
assert item.sku == "ABC-001"
assert item.quantity == 8
assert item.price == 19.95
print(item.__dict__)

try:
    item.quantity = 2.5
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

{'_sku': 'ABC-001', '_quantity': 8, '_price': 19.95}
TypeError: quantity must be int; got float


# Problem 3 — Reject `bool` when an integer is required

Python has a subtle type-system detail: `isinstance(True, int)` is `True`.

Implement `StrictInteger` so booleans are rejected even though they are integer subclasses.

## Solution 3

In [4]:
class StrictInteger:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{self.name} must be a non-bool integer")
        setattr(instance, self.storage_name, value)


class InventoryLine:
    units = StrictInteger()

    def __init__(self, units):
        self.units = units


ok = InventoryLine(12)
assert ok.units == 12

for bad in (True, False, 3.5, "7"):
    try:
        InventoryLine(bad)
    except TypeError as exc:
        print(repr(bad), "->", exc)

True -> units must be a non-bool integer
False -> units must be a non-bool integer
3.5 -> units must be a non-bool integer
'7' -> units must be a non-bool integer


# Problem 4 — Add numeric range validation

Create a descriptor `Number` supporting type validation, optional min/max values, inclusive/exclusive bounds, and meaningful errors.

## Solution 4

In [5]:
class Number:
    def __init__(self, expected_type, *, min_value=None, max_value=None,
                 min_inclusive=True, max_inclusive=True):
        self.expected_type = expected_type
        self.min_value = min_value
        self.max_value = max_value
        self.min_inclusive = min_inclusive
        self.max_inclusive = max_inclusive

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} must be {self.expected_type.__name__}")

        if self.min_value is not None:
            ok = value >= self.min_value if self.min_inclusive else value > self.min_value
            relation = ">=" if self.min_inclusive else ">"
            if not ok:
                raise ValueError(f"{self.name} must be {relation} {self.min_value}")

        if self.max_value is not None:
            ok = value <= self.max_value if self.max_inclusive else value < self.max_value
            relation = "<=" if self.max_inclusive else "<"
            if not ok:
                raise ValueError(f"{self.name} must be {relation} {self.max_value}")

        setattr(instance, self.storage_name, value)


class Sensor:
    temperature = Number(float, min_value=-273.15)
    humidity = Number(float, min_value=0.0, max_value=100.0)

    def __init__(self, temperature, humidity):
        self.temperature = temperature
        self.humidity = humidity


s = Sensor(21.5, 47.0)
assert s.temperature == 21.5
assert s.humidity == 47.0

for name, value in [("temperature", -300.0), ("humidity", -1.0), ("humidity", 101.0)]:
    try:
        setattr(s, name, value)
    except (TypeError, ValueError) as exc:
        print(f"{name}={value!r} -> {exc}")

temperature=-300.0 -> temperature must be >= -273.15
humidity=-1.0 -> humidity must be >= 0.0
humidity=101.0 -> humidity must be <= 100.0


# Problem 5 — Descriptor deletion semantics

Enhance a descriptor with `__delete__`. Reading or deleting a missing value should raise `AttributeError`.

## Solution 5

In [6]:
class Managed:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if not hasattr(instance, self.storage_name):
            raise AttributeError(f"{type(instance).__name__}.{self.name} is not set")
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        setattr(instance, self.storage_name, value)

    def __delete__(self, instance):
        if not hasattr(instance, self.storage_name):
            raise AttributeError(f"{type(instance).__name__}.{self.name} is not set")
        delattr(instance, self.storage_name)


class Record:
    value = Managed()

r = Record()
r.value = 100
assert r.value == 100
del r.value
assert "_value" not in r.__dict__

try:
    print(r.value)
except AttributeError as exc:
    print("read:", exc)

read: Record.value is not set


# Problem 6 — Data descriptor precedence vs instance dictionary

Demonstrate that a data descriptor takes precedence over a same-named key in the instance dictionary, then compare it with a non-data descriptor.

## Solution 6

In [7]:
class DataDescriptor:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return "from data descriptor"

    def __set__(self, instance, value):
        instance.__dict__["_managed"] = value


class NonDataDescriptor:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return "from non-data descriptor"


class Demo:
    data = DataDescriptor()
    nondata = NonDataDescriptor()


d = Demo()
d.__dict__["data"] = "instance dictionary value"
d.__dict__["nondata"] = "instance dictionary value"

assert d.data == "from data descriptor"
assert d.nondata == "instance dictionary value"
print("data   ->", d.data)
print("nondata->", d.nondata)

data   -> from data descriptor
nondata-> instance dictionary value


### Lookup principle

Simplified instance attribute precedence:

1. data descriptor on the class/MRO
2. instance dictionary
3. non-data descriptor or ordinary class attribute
4. `__getattr__`, if defined

# Problem 7 — Make a descriptor work with `__slots__`

Use explicit slot-backed storage for a class without `__dict__`.

## Solution 7

In [8]:
class SlotBacked:
    def __init__(self, storage_name):
        self.storage_name = storage_name

    def __set_name__(self, owner, name):
        self.public_name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return getattr(instance, self.storage_name)
        except AttributeError:
            raise AttributeError(
                f"{type(instance).__name__}.{self.public_name} is not initialized"
            ) from None

    def __set__(self, instance, value):
        setattr(instance, self.storage_name, value)


class SlottedPoint:
    __slots__ = ("_x", "_y")
    x = SlotBacked("_x")
    y = SlotBacked("_y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


p = SlottedPoint(10, 20)
assert p.x == 10 and p.y == 20
assert not hasattr(p, "__dict__")
print(p.x, p.y)
print("has __dict__?", hasattr(p, "__dict__"))

10 20
has __dict__? False


# Problem 8 — External descriptor storage and the memory-leak trap

A descriptor can keep a dictionary mapping instance -> value, but a normal dictionary keeps strong references to its keys. Prove that deleting the application reference may not destroy the instance.

## Solution 8

In [9]:
import gc
import weakref

class LeakyValue:
    def __init__(self):
        self.values = {}

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return self.values[instance]

    def __set__(self, instance, value):
        self.values[instance] = value


class LeakyModel:
    x = LeakyValue()

obj = LeakyModel()
obj.x = 42
watch = weakref.ref(obj)
assert len(LeakyModel.x.values) == 1

del obj
gc.collect()

print("Object still alive?", watch() is not None)
print("Descriptor entries:", len(LeakyModel.x.values))
assert watch() is not None

Object still alive? True
Descriptor entries: 1


# Problem 9 — Fix external storage with `WeakKeyDictionary`

Use `weakref.WeakKeyDictionary` so descriptor storage does not keep managed instances alive.

## Solution 9

In [10]:
import gc
import weakref

class WeakValue:
    def __init__(self):
        self.values = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self.values[instance]
        except KeyError:
            raise AttributeError("value has not been initialized") from None

    def __set__(self, instance, value):
        self.values[instance] = value

    def __delete__(self, instance):
        try:
            del self.values[instance]
        except KeyError:
            raise AttributeError("value has not been initialized") from None


class SafeModel:
    x = WeakValue()

obj = SafeModel()
obj.x = 99
watch = weakref.ref(obj)
assert len(SafeModel.x.values) == 1

del obj
gc.collect()

print("Object alive?", watch() is not None)
print("Descriptor entries:", len(SafeModel.x.values))
assert watch() is None
assert len(SafeModel.x.values) == 0

Object alive? False
Descriptor entries: 0


# Problem 10 — Weak references + `__slots__`

Show a slotted class that cannot be weak-referenced, then repair it by adding `__weakref__` to `__slots__`.

## Solution 10

In [11]:
import weakref

class NoWeakRefs:
    __slots__ = ()

a = NoWeakRefs()
try:
    weakref.ref(a)
except TypeError as exc:
    print("Expected failure:", exc)


class SupportsWeakRefs:
    __slots__ = ("__weakref__",)

b = SupportsWeakRefs()
ref = weakref.ref(b)
assert ref() is b
print("Weak reference works:", ref() is b)

Expected failure: cannot create weak reference to 'NoWeakRefs' object
Weak reference works: True


In [12]:
class WeakStored:
    def __init__(self):
        self.values = weakref.WeakKeyDictionary()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self.values[instance]
        except KeyError:
            raise AttributeError(self.name) from None

    def __set__(self, instance, value):
        self.values[instance] = value


class SlottedUser:
    __slots__ = ("__weakref__",)
    score = WeakStored()

u = SlottedUser()
u.score = 88
assert u.score == 88
assert len(SlottedUser.score.values) == 1
print(u.score)

88


# Problem 11 — Hashability constraints of `WeakKeyDictionary`

Construct an object that is weak-referenceable but intentionally unhashable, then observe why it cannot be a `WeakKeyDictionary` key.

## Solution 11

In [13]:
import weakref

class Unhashable:
    __hash__ = None

u = Unhashable()
assert weakref.ref(u)() is u
wk = weakref.WeakKeyDictionary()

try:
    wk[u] = "value"
except TypeError as exc:
    print("Expected failure:", exc)

Expected failure: unhashable type: 'Unhashable'


### Design note

External weak-key storage is useful when the descriptor must not modify the instance namespace and the target supports weak references and hashing. Instance-side storage is often simpler when a suitable `__dict__` or slot is available.

# Problem 12 — Detect descriptor reuse under multiple attribute names

If the same descriptor instance is assigned to two different names, repeated `__set_name__` calls can overwrite its metadata. Build a descriptor that rejects such reuse.

## Solution 12

In [14]:
class SingleBindingDescriptor:
    def __init__(self):
        self.owner = None
        self.name = None

    def __set_name__(self, owner, name):
        if self.owner is None:
            self.owner = owner
            self.name = name
            return
        if self.owner is not owner or self.name != name:
            raise RuntimeError("Descriptor instance reused under a different binding")

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__.get(f"_{self.name}")

    def __set__(self, instance, value):
        instance.__dict__[f"_{self.name}"] = value


class Good:
    x = SingleBindingDescriptor()

g = Good(); g.x = 5
assert g.x == 5
print(g.x)

# Uncomment to see class-creation failure:
# shared = SingleBindingDescriptor()
# class Bad:
#     x = shared
#     y = shared

5


# Problem 13 — A reusable validation framework

Create a base descriptor handling storage, with subclasses implementing `validate`. Build `Positive`, `NonEmptyString`, and `OneOf`.

## Solution 13

In [15]:
class Validated:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if not hasattr(instance, self.storage_name):
            raise AttributeError(f"{type(instance).__name__}.{self.name} is not initialized")
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        self.validate(value)
        setattr(instance, self.storage_name, value)

    def validate(self, value):
        raise NotImplementedError


class Positive(Validated):
    def validate(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} must be numeric")
        if value <= 0:
            raise ValueError(f"{self.name} must be > 0")


class NonEmptyString(Validated):
    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be str")
        if not value.strip():
            raise ValueError(f"{self.name} cannot be blank")


class OneOf(Validated):
    def __init__(self, *allowed):
        if not allowed:
            raise ValueError("OneOf requires at least one allowed value")
        self.allowed = frozenset(allowed)

    def validate(self, value):
        if value not in self.allowed:
            allowed = ", ".join(map(repr, sorted(self.allowed, key=repr)))
            raise ValueError(f"{self.name} must be one of: {allowed}")


class Employee:
    name = NonEmptyString()
    salary = Positive()
    level = OneOf("junior", "mid", "senior")

    def __init__(self, name, salary, level):
        self.name = name
        self.salary = salary
        self.level = level


e = Employee("Ada", 150_000, "senior")
assert e.name == "Ada"
for attr, value in [("name", "   "), ("salary", 0), ("level", "principal")]:
    try:
        setattr(e, attr, value)
    except (TypeError, ValueError) as exc:
        print(f"{attr}={value!r}: {exc}")

name='   ': name cannot be blank
salary=0: salary must be > 0
level='principal': level must be one of: 'junior', 'mid', 'senior'


# Problem 14 — Coercing descriptor vs validating descriptor

Compare an `int(value)` coercing descriptor with a strict integer descriptor for several inputs.

## Solution 14

In [16]:
class CoercedInteger:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        try:
            converted = int(value)
        except (TypeError, ValueError) as exc:
            raise TypeError(f"{self.name} cannot be converted to int") from exc
        setattr(instance, self.storage_name, converted)


class StrictInteger2:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{self.name} must be an integer")
        setattr(instance, self.storage_name, value)


class CoerceDemo:
    value = CoercedInteger()

class StrictDemo:
    value = StrictInteger2()

for value in [10, 10.8, "42", "abc", True]:
    print("\nInput:", repr(value))
    c = CoerceDemo()
    try:
        c.value = value
        print("  coerced ->", c.value)
    except Exception as exc:
        print("  coerced ->", type(exc).__name__, exc)

    s = StrictDemo()
    try:
        s.value = value
        print("  strict  ->", s.value)
    except Exception as exc:
        print("  strict  ->", type(exc).__name__, exc)


Input: 10
  coerced -> 10
  strict  -> 10

Input: 10.8
  coerced -> 10
  strict  -> TypeError value must be an integer

Input: '42'
  coerced -> 42
  strict  -> TypeError value must be an integer

Input: 'abc'
  coerced -> TypeError value cannot be converted to int
  strict  -> TypeError value must be an integer

Input: True
  coerced -> 1
  strict  -> TypeError value must be an integer


# Problem 15 — Read-only descriptor

Implement a descriptor that computes a value from an instance and rejects assignment. Use it for rectangle area.

## Solution 15

In [17]:
class ReadOnly:
    def __init__(self, function):
        self.function = function
        self.__doc__ = getattr(function, "__doc__")

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return self.function(instance)

    def __set__(self, instance, value):
        raise AttributeError(f"{self.name} is read-only")


class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @ReadOnly
    def area(self):
        return self.width * self.height

r = Rectangle(3, 4)
assert r.area == 12
try:
    r.area = 999
except AttributeError as exc:
    print(exc)

area is read-only


# Problem 16 — Cached non-data descriptor with invalidation

Create a non-data descriptor that computes once and stores the result in the instance dictionary under the same public name. Then invalidate it by deleting that instance key.

## Solution 16

In [18]:
class cached:
    def __init__(self, func):
        self.func = func
        self.name = func.__name__
        self.__doc__ = getattr(func, "__doc__")

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        value = self.func(instance)
        instance.__dict__[self.name] = value
        return value


class Expensive:
    def __init__(self, n):
        self.n = n
        self.calls = 0

    @cached
    def result(self):
        self.calls += 1
        return sum(i * i for i in range(self.n))


e = Expensive(10_000)
first = e.result
second = e.result
assert first == second and e.calls == 1

del e.__dict__["result"]
third = e.result
assert third == first and e.calls == 2
print("calls:", e.calls)

calls: 2


# Problem 17 — Descriptor inheritance and owner argument

Build a descriptor that reports where it was declared, the actual instance type, and the owner through which it was accessed.

## Solution 17

In [19]:
class OwnerAware:
    def __set_name__(self, owner, name):
        self.declared_on = owner
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return {
                "descriptor_name": self.name,
                "declared_on": self.declared_on.__name__,
                "accessed_through": owner.__name__ if owner else None,
            }
        return {
            "descriptor_name": self.name,
            "declared_on": self.declared_on.__name__,
            "instance_type": type(instance).__name__,
            "accessed_through": owner.__name__ if owner else None,
        }

class Base:
    info = OwnerAware()

class Child(Base):
    pass

print(Base.info)
print(Child.info)
print(Base().info)
print(Child().info)

{'descriptor_name': 'info', 'declared_on': 'Base', 'accessed_through': 'Base'}
{'descriptor_name': 'info', 'declared_on': 'Base', 'accessed_through': 'Child'}
{'descriptor_name': 'info', 'declared_on': 'Base', 'instance_type': 'Base', 'accessed_through': 'Base'}
{'descriptor_name': 'info', 'declared_on': 'Base', 'instance_type': 'Child', 'accessed_through': 'Child'}


# Problem 18 — Override a descriptor in a subclass

Predict and verify what happens when a subclass replaces a descriptor with a plain class attribute or another descriptor. Use `inspect.getattr_static` to inspect without triggering binding.

## Solution 18

In [20]:
import inspect

class LabelDescriptor:
    def __init__(self, label):
        self.label = label

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return f"{self.label}:{type(instance).__name__}"

class Parent:
    value = LabelDescriptor("parent")

class PlainChild(Parent):
    value = "plain override"

class DescriptorChild(Parent):
    value = LabelDescriptor("child")

print("Parent instance:", Parent().value)
print("PlainChild instance:", PlainChild().value)
print("DescriptorChild instance:", DescriptorChild().value)
print("Raw Parent.value:", inspect.getattr_static(Parent, "value"))
print("Raw PlainChild.value:", inspect.getattr_static(PlainChild, "value"))
print("Raw DescriptorChild.value:", inspect.getattr_static(DescriptorChild, "value"))

Parent instance: parent:Parent
PlainChild instance: plain override
DescriptorChild instance: child:DescriptorChild
Raw Parent.value: <__main__.LabelDescriptor object at 0x000002A841895160>
Raw PlainChild.value: plain override
Raw DescriptorChild.value: <__main__.LabelDescriptor object at 0x000002A84185C690>


# Problem 19 — Audit logging descriptor

Record successful assignments without keeping managed instances alive solely because of the audit log.

## Solution 19

In [21]:
import weakref
_MISSING = object()

class Audited:
    def __init__(self):
        self.events = []

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if not hasattr(instance, self.storage_name):
            raise AttributeError(self.name)
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        old = getattr(instance, self.storage_name, _MISSING)
        setattr(instance, self.storage_name, value)
        try:
            obj_ref = weakref.ref(instance)
        except TypeError:
            obj_ref = None
        self.events.append({
            "object_id": id(instance),
            "object_ref": obj_ref,
            "attribute": self.name,
            "old": old,
            "new": value,
        })

class Account:
    balance = Audited()
    def __init__(self, balance):
        self.balance = balance


a = Account(100)
a.balance = 150
a.balance = 125
for event in Account.balance.events:
    printable = dict(event)
    printable["old"] = "<MISSING>" if printable["old"] is _MISSING else printable["old"]
    print(printable)

{'object_id': 2921677280256, 'object_ref': <weakref at 0x000002A8418CA520; to 'Account' at 0x000002A841895400>, 'attribute': 'balance', 'old': '<MISSING>', 'new': 100}
{'object_id': 2921677280256, 'object_ref': <weakref at 0x000002A8418CA520; to 'Account' at 0x000002A841895400>, 'attribute': 'balance', 'old': 100, 'new': 150}
{'object_id': 2921677280256, 'object_ref': <weakref at 0x000002A8418CA520; to 'Account' at 0x000002A841895400>, 'attribute': 'balance', 'old': 150, 'new': 125}


# Problem 20 — Thread-safe descriptor storage

Build a weak-key descriptor using `RLock`; lock reads, writes, deletes, and tracking operations.

## Solution 20

In [22]:
import threading
import weakref

class ThreadSafeWeakValue:
    def __init__(self):
        self._values = weakref.WeakKeyDictionary()
        self._lock = threading.RLock()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        with self._lock:
            try:
                return self._values[instance]
            except KeyError:
                raise AttributeError(self.name) from None

    def __set__(self, instance, value):
        with self._lock:
            self._values[instance] = value

    def __delete__(self, instance):
        with self._lock:
            try:
                del self._values[instance]
            except KeyError:
                raise AttributeError(self.name) from None

    def tracked_instances(self):
        with self._lock:
            return len(self._values)


class SharedState:
    value = ThreadSafeWeakValue()

items = [SharedState() for _ in range(100)]

def worker(start, stop):
    for i in range(start, stop):
        items[i].value = i

threads = [
    threading.Thread(target=worker, args=(0, 50)),
    threading.Thread(target=worker, args=(50, 100)),
]
for t in threads: t.start()
for t in threads: t.join()

assert all(items[i].value == i for i in range(100))
assert SharedState.value.tracked_instances() == 100
print("tracked:", SharedState.value.tracked_instances())

tracked: 100


# Problem 21 — Cross-field validation

Build an interval whose invariant is `start <= end`. Assignment to either endpoint must preserve the invariant, while initialization must avoid false failure before both endpoints exist.

## Solution 21

In [23]:
class Endpoint:
    def __init__(self, *, is_start):
        self.is_start = is_start

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} must be numeric")

        other_storage = "_end" if self.is_start else "_start"
        if hasattr(instance, other_storage):
            other = getattr(instance, other_storage)
            if self.is_start and value > other:
                raise ValueError("start cannot be greater than end")
            if not self.is_start and value < other:
                raise ValueError("end cannot be less than start")

        setattr(instance, self.storage_name, value)


class Interval:
    start = Endpoint(is_start=True)
    end = Endpoint(is_start=False)

    def __init__(self, start, end):
        self._start = start
        self.end = end
        self.start = self._start

interval = Interval(10, 20)
assert interval.start == 10 and interval.end == 20
interval.start = 15
interval.end = 25

for attr, value in [("start", 30), ("end", 5)]:
    try:
        setattr(interval, attr, value)
    except ValueError as exc:
        print(f"{attr}={value}: {exc}")

start=30: start cannot be greater than end
end=5: end cannot be less than start


# Problem 22 — Descriptor factory

Write a factory function returning a descriptor class specialized for a type.

## Solution 22

In [24]:
def typed_field(expected_type):
    class TypedField:
        def __set_name__(self, owner, name):
            self.name = name
            self.storage_name = f"_{name}"

        def __get__(self, instance, owner=None):
            if instance is None:
                return self
            return getattr(instance, self.storage_name)

        def __set__(self, instance, value):
            if not isinstance(value, expected_type):
                raise TypeError(f"{self.name} must be {expected_type.__name__}")
            setattr(instance, self.storage_name, value)

    TypedField.__name__ = f"{expected_type.__name__.title()}Field"
    return TypedField

IntegerField = typed_field(int)
StringField = typed_field(str)

class User:
    age = IntegerField()
    name = StringField()
    def __init__(self, age, name):
        self.age = age
        self.name = name

user = User(36, "Grace")
assert user.age == 36 and user.name == "Grace"
print(user.__dict__)

{'_age': 36, '_name': 'Grace'}


# Problem 23 — Automatically clone descriptor prototypes for reuse

A framework can use a metaclass to clone `Field` prototypes so the same prototype object can be mentioned in multiple class bodies without sharing owner/name metadata.

## Solution 23

In [25]:
import copy

class PrototypeField:
    def __init__(self, *, validator=None):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if self.validator is not None:
            self.validator(value)
        setattr(instance, self.storage_name, value)

class CloneFieldsMeta(type):
    def __new__(mcls, name, bases, namespace):
        ns = dict(namespace)
        for attr_name, value in namespace.items():
            if isinstance(value, PrototypeField):
                ns[attr_name] = copy.copy(value)
        return super().__new__(mcls, name, bases, ns)

def positive(value):
    if value <= 0:
        raise ValueError("must be positive")

prototype = PrototypeField(validator=positive)

class A(metaclass=CloneFieldsMeta):
    x = prototype

class B(metaclass=CloneFieldsMeta):
    y = prototype

assert A.__dict__["x"] is not B.__dict__["y"]
a = A(); b = B(); a.x = 10; b.y = 20
print(a.x, b.y)

10 20


# Problem 24 — Introspect descriptors across a class hierarchy

Write `find_descriptors(cls)` that walks the MRO, avoids triggering `__get__`, reports where descriptors were declared, and lets subclass attributes override base attributes.

## Solution 24

In [26]:
def find_descriptors(cls):
    found = {}
    for base in reversed(cls.__mro__):
        for name, value in vars(base).items():
            if hasattr(type(value), "__get__"):
                found[name] = {
                    "declared_on": base.__name__,
                    "object": value,
                    "type": type(value).__name__,
                }
    return found

class DescriptorA:
    def __get__(self, instance, owner=None): return "A"
class DescriptorB:
    def __get__(self, instance, owner=None): return "B"

class Root:
    x = DescriptorA()
class Mid(Root):
    y = DescriptorB()
class Leaf(Mid):
    x = DescriptorB()

results = find_descriptors(Leaf)
for name in sorted(results):
    if not name.startswith("__"):
        print(name, "->", results[name])

x -> {'declared_on': 'Leaf', 'object': <__main__.DescriptorB object at 0x000002A84185D810>, 'type': 'DescriptorB'}
y -> {'declared_on': 'Mid', 'object': <__main__.DescriptorB object at 0x000002A8418963C0>, 'type': 'DescriptorB'}


# Problem 25 — Benchmark two storage strategies

Compare instance attribute storage with `WeakKeyDictionary` storage using `timeit`. This is an intuition-building microbenchmark, not a universal performance claim.

## Solution 25

In [27]:
import timeit
import weakref

class InstanceStored:
    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}"
    def __get__(self, instance, owner=None):
        if instance is None: return self
        return getattr(instance, self.storage_name)
    def __set__(self, instance, value):
        setattr(instance, self.storage_name, value)

class WeakStored2:
    def __init__(self):
        self.values = weakref.WeakKeyDictionary()
    def __get__(self, instance, owner=None):
        if instance is None: return self
        return self.values[instance]
    def __set__(self, instance, value):
        self.values[instance] = value

class A2: x = InstanceStored()
class B2: x = WeakStored2()

a2 = A2(); b2 = B2(); a2.x = 1; b2.x = 1
print("instance read :", timeit.timeit("a2.x", number=200_000, globals=globals()))
print("weak read     :", timeit.timeit("b2.x", number=200_000, globals=globals()))
print("instance write:", timeit.timeit("a2.x = 2", number=200_000, globals=globals()))
print("weak write    :", timeit.timeit("b2.x = 2", number=200_000, globals=globals()))

instance read : 0.037402199988719076
weak read     : 0.05849130000569858
instance write: 0.039218599995365366
weak write    : 0.085143500007689


# Problem 26 — Final integrated challenge: production-style field system

Build a mini field framework with automatic naming, type validation, nullable values, defaults, custom validators, read-only fields, deletion, `repr`, and class-level introspection.

## Solution 26

In [28]:
_MISSING2 = object()

class Field:
    def __init__(self, expected_type, *, default=_MISSING2, nullable=False,
                 validator=None, read_only=False):
        self.expected_type = expected_type
        self.default = default
        self.nullable = nullable
        self.validator = validator
        self.read_only = read_only
        self.owner = None
        self.name = None
        self.storage_name = None

    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name
        self.storage_name = f"_{name}"

    def __repr__(self):
        owner_name = self.owner.__name__ if self.owner else "?"
        type_name = getattr(self.expected_type, "__name__", repr(self.expected_type))
        return (f"<Field {owner_name}.{self.name}: {type_name}, "
                f"nullable={self.nullable}, read_only={self.read_only}>")

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if hasattr(instance, self.storage_name):
            return getattr(instance, self.storage_name)
        if self.default is not _MISSING2:
            return self.default() if callable(self.default) else self.default
        raise AttributeError(f"{type(instance).__name__}.{self.name} is not initialized")

    def __set__(self, instance, value):
        if self.read_only and hasattr(instance, self.storage_name):
            raise AttributeError(f"{self.name} is read-only after initialization")

        if value is None:
            if not self.nullable:
                raise TypeError(f"{self.name} cannot be None")
            setattr(instance, self.storage_name, None)
            return

        if isinstance(value, bool) and self.expected_type is int:
            raise TypeError(f"{self.name} must be int, not bool")

        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be {self.expected_type.__name__}; "
                f"got {type(value).__name__}"
            )

        if self.validator is not None:
            self.validator(value)
        setattr(instance, self.storage_name, value)

    def __delete__(self, instance):
        if self.read_only:
            raise AttributeError(f"{self.name} cannot be deleted")
        if not hasattr(instance, self.storage_name):
            raise AttributeError(f"{self.name} is not set")
        delattr(instance, self.storage_name)


def in_range(low, high):
    def validate(value):
        if not low <= value <= high:
            raise ValueError(f"value must be between {low} and {high}, inclusive")
    return validate


def non_blank(value):
    if not value.strip():
        raise ValueError("value cannot be blank")


def contains_at_sign(value):
    if "@" not in value:
        raise ValueError("email must contain '@'")


class Model:
    @classmethod
    def fields(cls):
        result = {}
        for base in reversed(cls.__mro__):
            for name, value in vars(base).items():
                if isinstance(value, Field):
                    result[name] = value
        return result


class Customer(Model):
    name = Field(str, validator=non_blank)
    age = Field(int, validator=in_range(0, 130))
    email = Field(str, validator=contains_at_sign)
    customer_id = Field(int, read_only=True)

    def __init__(self, *, name, age, email, customer_id):
        self.name = name
        self.age = age
        self.email = email
        self.customer_id = customer_id


customer = Customer(name="Lin", age=31, email="lin@example.com", customer_id=1001)
assert customer.name == "Lin"
assert customer.age == 31
assert customer.email == "lin@example.com"
assert customer.customer_id == 1001
print(customer.__dict__)
for name, field in Customer.fields().items():
    print(name, "->", field)

try:
    customer.customer_id = 2002
except AttributeError as exc:
    print("read-only check:", exc)
try:
    customer.age = 500
except ValueError as exc:
    print("age check:", exc)
try:
    customer.email = "invalid-email"
except ValueError as exc:
    print("email check:", exc)

{'_name': 'Lin', '_age': 31, '_email': 'lin@example.com', '_customer_id': 1001}
name -> <Field Customer.name: str, nullable=False, read_only=False>
age -> <Field Customer.age: int, nullable=False, read_only=False>
email -> <Field Customer.email: str, nullable=False, read_only=False>
customer_id -> <Field Customer.customer_id: int, nullable=False, read_only=True>
read-only check: customer_id is read-only after initialization
age check: value must be between 0 and 130, inclusive
email check: email must contain '@'


# Problem 27 — Compact test suite for the final framework

Cover valid initialization, bad types, bool-vs-int, range validation, blank strings, invalid email, read-only assignment, class access, and field introspection.

## Solution 27

In [29]:
def assert_raises(exc_type, func, *args, **kwargs):
    try:
        func(*args, **kwargs)
    except exc_type:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised")

c = Customer(name="Maya", age=40, email="maya@example.org", customer_id=7)
assert c.name == "Maya"
assert_raises(TypeError, setattr, c, "age", "forty")
assert_raises(TypeError, setattr, c, "age", True)
assert_raises(ValueError, setattr, c, "age", 131)
assert_raises(ValueError, setattr, c, "name", "   ")
assert_raises(ValueError, setattr, c, "email", "not-an-email")
assert_raises(AttributeError, setattr, c, "customer_id", 999)
assert isinstance(Customer.age, Field)
assert set(Customer.fields()) == {"name", "age", "email", "customer_id"}
print("All tests passed.")

All tests passed.


# Extra challenge prompts

1. Add an explicit `default_factory` separate from literal defaults.
2. Add a `choices=` constraint to `Field`.
3. Add a `transform=` hook before validation.
4. Add per-field documentation and generate a schema dictionary.
5. Make `Model.__repr__` automatically display all fields.
6. Add a `frozen=True` model option.
7. Support JSON aliases such as `customerId -> customer_id`.
8. Add a descriptor that counts reads/writes without keeping instances alive.
9. Design external storage that can support unhashable instances.
10. Compare descriptors with `property`, `dataclasses`, and custom `__setattr__`.
11. Add `reset(instance)` to restore defaults.
12. Reject duplicate private storage names during class creation.
13. Explore multiple inheritance conflicts.
14. Explore interaction with an overridden `__getattribute__`.
15. Implement a weak-reference-safe observer descriptor.

# Key takeaways

- Descriptor objects normally live on the class and are shared.
- Per-instance values therefore need a deliberate storage strategy.
- A fixed storage key causes collisions when a descriptor class is used for multiple fields.
- `__set_name__` removes duplicated naming boilerplate.
- Instance-side storage is often the simplest design when available.
- Restrictive `__slots__` changes the available storage strategies.
- A normal dictionary keyed by managed instances can keep those instances alive.
- `weakref.WeakKeyDictionary` avoids that strong-reference leak, but keys must support weak references and hashing.
- Data descriptors have higher lookup precedence than the instance dictionary.
- Non-data descriptors can intentionally allow instance values to shadow them.
- Mature descriptor designs should consider validation semantics, deletion, inheritance, introspection, storage lifetime, and concurrency.